# Data Engineering

This notebook prepares the dataset for machine learning by performing data cleaning, feature engineering and pipeline construction using PySpark.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Project Configuration

In [ ]:
import os, subprocess, json


BASE = "/content/drive/MyDrive/7006SCN"
DATA = f"{BASE}/data"
PROC = f"{BASE}/processed"
MODELS = f"{BASE}/models"
META = f"{BASE}/metadata"
OUTPUTS = f"{BASE}/outputs"

# ------------------------------------------------------------------
# Artefact paths — referenced by SAVE (producer) and LOAD (consumer)
# ------------------------------------------------------------------
RAW_DATA_PATH = f"{DATA}/taxi.csv"

PROC_TRAIN = f"{PROC}/training.parquet"
PROC_TEST = f"{PROC}/test.parquet"

PIPELINE_PATH = f"{MODELS}/preprocessing_pipeline"

LR_MODEL_PATH = f"{MODELS}/lr_model"
RF_MODEL_PATH = f"{MODELS}/rf_model"
GBT_MODEL_PATH = f"{MODELS}/gbt_model"

TASK1_META = f"{META}/task1_metadata.json"
TASK2_META = f"{META}/task2_metadata.json"
TASK3_META = f"{META}/task3_metadata.json"


for p in [DATA, PROC, MODELS, META, OUTPUTS]:
    os.makedirs(p, exist_ok=True)

def verify_exists(path, label=""):
    """subprocess verify — prints ls -lh for the path"""

    result = subprocess.run(
        ["ls", "-lh", path],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print(f"✓ {label or path}:")
        print(result.stdout.strip())
    else:
        raise FileNotFoundError(
            f"NOT FOUND: {path}\n{result.stderr}"
        )

print("Shared constants loaded ✓")

Shared constants loaded ✓


##Installing PySpark

In [ ]:
# ----------------------------------
# Install pyspark
# ----------------------------------

!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,059 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Fetched 13.8 MB in 5s (2,765 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
30 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping 

# Initialising Spark session

In [ ]:
# ----------------------------------
# Initialising new spark session
# ----------------------------------

from pyspark.sql import SparkSession
import time

spark = (
    SparkSession.builder
    .appName("7006SCN_ME_16882940_Task_2")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Session started:", spark.sparkContext.applicationId)


Spark version: 4.0.2
Session started: local-1781446802187


# Uploading/reading the csv file

In [ ]:
raw_df = spark.read.csv(
    RAW_DATA_PATH,
    header=True,
    inferSchema=False
)

# Preprocessing

In [ ]:
# ----------------------------------
# Cleaning the column names
# ----------------------------------

raw_df = raw_df.toDF(*[c.strip().replace(" ", "_") for c in raw_df.columns])
print(raw_df.columns)

['Trip_ID', 'Taxi_ID', 'Trip_Start_Timestamp', 'Trip_End_Timestamp', 'Trip_Seconds', 'Trip_Miles', 'Pickup_Census_Tract', 'Dropoff_Census_Tract', 'Pickup_Community_Area', 'Dropoff_Community_Area', 'Fare', 'Tips', 'Tolls', 'Extras', 'Trip_Total', 'Payment_Type', 'Company', 'Pickup_Centroid_Latitude', 'Pickup_Centroid_Longitude', 'Pickup_Centroid_Location', 'Dropoff_Centroid_Latitude', 'Dropoff_Centroid_Longitude', 'Dropoff_Centroid__Location']


In [ ]:
# ----------------------------------
# Creating new data frame with only the necessary features
# ----------------------------------

raw_df = raw_df.select(
    "Trip_Seconds",
    "Trip_Miles",
    "Trip_Start_Timestamp",
    "Payment_Type",
    "Company",
    "Pickup_Community_Area",
    "Dropoff_Community_Area",
    "Pickup_Centroid_Latitude",
    "Pickup_Centroid_Longitude",
    "Dropoff_Centroid_Latitude",
    "Dropoff_Centroid_Longitude"
)

raw_df.show(10)

+------------+----------+--------------------+------------+--------------------+---------------------+----------------------+------------------------+-------------------------+-------------------------+--------------------------+
|Trip_Seconds|Trip_Miles|Trip_Start_Timestamp|Payment_Type|             Company|Pickup_Community_Area|Dropoff_Community_Area|Pickup_Centroid_Latitude|Pickup_Centroid_Longitude|Dropoff_Centroid_Latitude|Dropoff_Centroid_Longitude|
+------------+----------+--------------------+------------+--------------------+---------------------+----------------------+------------------------+-------------------------+-------------------------+--------------------------+
|         982|      0.81|12/31/2023 11:45:...| Credit Card|Chicago Independents|                    8|                     8|             41.89321636|             -87.63784421|             41.892507781|             -87.626214906|
|          12|      0.57|12/31/2023 11:45:...|        Cash|         5 Star Taxi|

In [ ]:
from pyspark.sql import functions as F

raw_df = raw_df.withColumn(
    "Trip_Seconds",
    F.regexp_replace("Trip_Seconds", ",", "").cast("int")
)

raw_df = raw_df.withColumn(
    "Trip_Miles",
    F.regexp_replace("Trip_Miles", ",", "").cast("double")
)

In [ ]:
# ----------------------------------
# Converting data types
# ----------------------------------

from pyspark.sql import types as T

# ----------------------------------
# Numeric columns
# ----------------------------------

raw_df = raw_df.withColumn(
    "Trip_Seconds",
    F.col("Trip_Seconds").cast(T.IntegerType())
).withColumn(
    "Trip_Miles",
    F.col("Trip_Miles").cast(T.DoubleType())
).withColumn(
    "Pickup_Centroid_Latitude",
    F.col("Pickup_Centroid_Latitude").cast(T.DoubleType())
).withColumn(
    "Pickup_Centroid_Longitude",
    F.col("Pickup_Centroid_Longitude").cast(T.DoubleType())
).withColumn(
    "Dropoff_Centroid_Latitude",
    F.col("Dropoff_Centroid_Latitude").cast(T.DoubleType())
).withColumn(
    "Dropoff_Centroid_Longitude",
    F.col("Dropoff_Centroid_Longitude").cast(T.DoubleType())
)

# ----------------------------------
# Timestamp column
# ----------------------------------

raw_df = raw_df.withColumn(
    "Trip_Start_Timestamp",
    F.to_timestamp(
        "Trip_Start_Timestamp",
        "MM/dd/yyyy hh:mm:ss a"
    )
)

# ----------------------------------
# Double checking
# ----------------------------------

raw_df.printSchema()

root
 |-- Trip_Seconds: integer (nullable = true)
 |-- Trip_Miles: double (nullable = true)
 |-- Trip_Start_Timestamp: timestamp (nullable = true)
 |-- Payment_Type: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Pickup_Community_Area: string (nullable = true)
 |-- Dropoff_Community_Area: string (nullable = true)
 |-- Pickup_Centroid_Latitude: double (nullable = true)
 |-- Pickup_Centroid_Longitude: double (nullable = true)
 |-- Dropoff_Centroid_Latitude: double (nullable = true)
 |-- Dropoff_Centroid_Longitude: double (nullable = true)



In [ ]:
# ----------------------------------
# Removing invalid values
# ----------------------------------

raw_df = (
    raw_df
    .filter(F.col("Trip_Seconds") > 0)
    .filter(F.col("Trip_Miles") > 0)
)

In [ ]:
# ----------------------------------
# Removing duplicates
# ----------------------------------

print("Rows before:", raw_df.count())

raw_df = raw_df.distinct()

print("Rows after:", raw_df.count())

Rows before: 11387084
Rows after: 11374998


In [ ]:
# -----------------------------------------------------------------
# Creating cleaned data frame by removing rows with missing values
# Comparing the row count before and after cleaning
# -----------------------------------------------------------------

before_count = raw_df.count()

clean_df = raw_df.na.drop()

after_count = clean_df.count()

print("Rows before cleaning:", before_count)
print("Rows after cleaning:", after_count)
print("Rows removed:", before_count - after_count)

Rows before cleaning: 11374998
Rows after cleaning: 9936804
Rows removed: 1438194


# Feature engineering

In [ ]:
# ----------------------------------
# Creating a column for "Hour_of_Day"
# ----------------------------------

clean_df = clean_df.withColumn(
    "Hour_of_Day",
    F.hour("Trip_Start_Timestamp")
)

# ----------------------------------
# Creating a column for "Day_of_Week"
# ----------------------------------
clean_df = clean_df.withColumn(
    "Day_of_Week",
    F.dayofweek("Trip_Start_Timestamp")
)

# ----------------------------------
# Creating a column for "Month"
# ----------------------------------

clean_df = clean_df.withColumn(
    "Month",
    F.month("Trip_Start_Timestamp") )

In [ ]:
# ----------------------------------
# Creating a rush hour flag
# ----------------------------------

clean_df = clean_df.withColumn(
    "Rush_Hour_Flag",
    F.when(
        ((F.hour("Trip_Start_Timestamp") >= 7) &
         (F.hour("Trip_Start_Timestamp") <= 9)) |
        ((F.hour("Trip_Start_Timestamp") >= 16) &
         (F.hour("Trip_Start_Timestamp") <= 18)), 1).otherwise(0)
)

In [ ]:
# ----------------------------------
# Repartiton and cache data frame
# ----------------------------------

clean_df = clean_df.repartition(200)
clean_df.cache()

print(f"Clean row count : {clean_df.count():,}")
print(f"Partitions      : {clean_df.rdd.getNumPartitions()}")
clean_df.show(5)

Clean row count : 9,936,804
Partitions      : 200
+------------+----------+--------------------+------------+--------------------+---------------------+----------------------+------------------------+-------------------------+-------------------------+--------------------------+-----------+-----------+-----+--------------+
|Trip_Seconds|Trip_Miles|Trip_Start_Timestamp|Payment_Type|             Company|Pickup_Community_Area|Dropoff_Community_Area|Pickup_Centroid_Latitude|Pickup_Centroid_Longitude|Dropoff_Centroid_Latitude|Dropoff_Centroid_Longitude|Hour_of_Day|Day_of_Week|Month|Rush_Hour_Flag|
+------------+----------+--------------------+------------+--------------------+---------------------+----------------------+------------------------+-------------------------+-------------------------+--------------------------+-----------+-----------+-----+--------------+
|        1320|      10.6| 2023-09-24 22:00:00|     Unknown|Taxi Affiliation ...|                   43|                     8|

In [ ]:
# ----------------------------------
# Check final schema
# ----------------------------------

clean_df.printSchema()


# ----------------------------------
# Check remaining nulll values
# ----------------------------------

null_counts = clean_df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in clean_df.columns]
)
null_counts.show()

root
 |-- Trip_Seconds: integer (nullable = true)
 |-- Trip_Miles: double (nullable = true)
 |-- Trip_Start_Timestamp: timestamp (nullable = true)
 |-- Payment_Type: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Pickup_Community_Area: string (nullable = true)
 |-- Dropoff_Community_Area: string (nullable = true)
 |-- Pickup_Centroid_Latitude: double (nullable = true)
 |-- Pickup_Centroid_Longitude: double (nullable = true)
 |-- Dropoff_Centroid_Latitude: double (nullable = true)
 |-- Dropoff_Centroid_Longitude: double (nullable = true)
 |-- Hour_of_Day: integer (nullable = true)
 |-- Day_of_Week: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Rush_Hour_Flag: integer (nullable = false)

+------------+----------+--------------------+------------+-------+---------------------+----------------------+------------------------+-------------------------+-------------------------+--------------------------+-----------+-----------+-----+--------------

# Data preperartion pipeline

In [ ]:
# ----------------------------------
# Installing libraries
# ----------------------------------

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

In [ ]:
# ----------------------------------
# Defining the feature groups
# ----------------------------------

NUMERIC_COLS = [
    "Trip_Miles",
    "Pickup_Centroid_Latitude",
    "Pickup_Centroid_Longitude",
    "Dropoff_Centroid_Latitude",
    "Dropoff_Centroid_Longitude",
    "Hour_of_Day",
    "Day_of_Week",
    "Month",
    "Rush_Hour_Flag"
]

CATEGORICAL_COLS = [
    "Payment_Type",
    "Company",
    "Pickup_Community_Area",
    "Dropoff_Community_Area"
]

TEMPORAL_COL = "Trip_Start_Timestamp"

LABEL_COL = "Trip_Seconds"

In [ ]:
clean_df.write.mode("overwrite").parquet(f"{PROC}/cleaned.parquet")

In [ ]:
# ----------------------------------
# Training and splitting data set
# ----------------------------------

train_df, test_df = clean_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [ ]:
# ----------------------------------
# Categorical pipeline
# ----------------------------------

payment_indexer = StringIndexer(
    inputCol="Payment_Type",
    outputCol="Payment_Type_Index",
    handleInvalid="keep"
)

company_indexer = StringIndexer(
    inputCol="Company",
    outputCol="Company_Index",
    handleInvalid="keep"
)

pickup_indexer = StringIndexer(
    inputCol="Pickup_Community_Area",
    outputCol="Pickup_Community_Area_Index",
    handleInvalid="keep"
)

dropoff_indexer = StringIndexer(
    inputCol="Dropoff_Community_Area",
    outputCol="Dropoff_Community_Area_Index",
    handleInvalid="keep"
)

payment_encoder = OneHotEncoder(
    inputCols=["Payment_Type_Index"],
    outputCols=["Payment_Type_Vec"]
)

company_encoder = OneHotEncoder(
    inputCols=["Company_Index"],
    outputCols=["Company_Vec"]
)

pickup_encoder = OneHotEncoder(
    inputCols=["Pickup_Community_Area_Index"],
    outputCols=["Pickup_Community_Area_Vec"]
)

dropoff_encoder = OneHotEncoder(
    inputCols=["Dropoff_Community_Area_Index"],
    outputCols=["Dropoff_Community_Area_Vec"]
)

In [ ]:
# ----------------------------------
# Numeric and categorical assembly
# ----------------------------------

assembler = VectorAssembler(
    inputCols=[
        "Trip_Miles",
        "Pickup_Centroid_Latitude",
        "Pickup_Centroid_Longitude",
        "Dropoff_Centroid_Latitude",
        "Dropoff_Centroid_Longitude",
        "Hour_of_Day",
        "Day_of_Week",
        "Month",
        "Rush_Hour_Flag",
        "Payment_Type_Vec",
        "Company_Vec",
        "Pickup_Community_Area_Vec",
        "Dropoff_Community_Area_Vec"
    ],
    outputCol="raw_features"
)

In [ ]:
# ----------------------------------
# Feature scaling
# ----------------------------------

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

In [ ]:
# ----------------------------------
# Feature engineerig pipeline
# ----------------------------------

fe_pipeline = Pipeline(
    stages=[
        payment_indexer,
        company_indexer,
        pickup_indexer,
        dropoff_indexer,
        payment_encoder,
        company_encoder,
        pickup_encoder,
        dropoff_encoder,
        assembler,
        scaler
    ]
)

In [ ]:
# ----------------------------------
# Insepecting engineered features
# ----------------------------------

preproc_model = fe_pipeline.fit(train_df)

train_fe = preproc_model.transform(train_df)

train_fe.select(
    "Trip_Miles",
    "Payment_Type",
    "Hour_of_Day",
    "features"
).show(10, truncate=False)

+----------+------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------
# Saving the fitted preprocessing PipelineModel
# ----------------------------------------------------

preproc_model.write().overwrite().save(PIPELINE_PATH)
print(f"Pipeline saved → {PIPELINE_PATH}")


# ----------------------------------------------------
# Saving transformed train/test splits as Parquet
# ----------------------------------------------------

train_df.write.parquet(PROC_TRAIN, mode="overwrite")
test_df.write.parquet(PROC_TEST, mode="overwrite")

print(f"Splits saved → {PROC_TRAIN}")


# ----------------------------------
# Saving Task2 metadata JSON
# ----------------------------------

task2_meta = {
    "feature_cols": NUMERIC_COLS + CATEGORICAL_COLS,
    "target_col": LABEL_COL,

    "pipeline_stages": [str(s) for s in preproc_model.stages],

    "train_rows": train_df.count(),
    "test_rows": test_df.count(),

    "train_partitions": train_df.rdd.getNumPartitions(),
    "test_partitions": test_df.rdd.getNumPartitions(),

    "null_counts": null_counts.collect()
}

# ----------------------------------
# Saving metadata json
# ----------------------------------

with open(TASK2_META, "w") as f:
    json.dump(task2_meta, f, indent=4)

print(f"Task 2 metadata saved → {TASK2_META}")

spark.stop()

Pipeline saved → /content/drive/MyDrive/7006SCN/models/preprocessing_pipeline
Splits saved → /content/drive/MyDrive/7006SCN/processed/training.parquet
Task 2 metadata saved → /content/drive/MyDrive/7006SCN/metadata/task2_metadata.json


## Summary

In this notebook, the Chicago Taxi Trips dataset was cleaned and prepared for machine learning. Missing values, duplicates and invalid records were removed, relevant temporal features were engineered, categorical variables were encoded and numerical features were standardised using a PySpark pipeline.

The processed dataset is now ready for training and comparing multiple machine learning models.